# 00 - Validate Prerequisites

Dry-run-first preflight for repository completeness, safe configuration, Python/Fabric runtime dependencies, workspace authorization, and feature/API prerequisites. It records evidence without exposing tokens or treating missing/unsupported capabilities as success.

In [ ]:
# PARAMETERS - supply runtime values through the notebook job API.
workspace_id = ''
capacity_reference = ''
artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
environment_name = 'dev'
dry_run = True
strict_mode = True

import importlib.util
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests


In [ ]:
ROOT = Path(artifact_root)
RESULTS = []
REQUIRED_PATHS = [
    'config/base/platform.json', 'config/environments/dev.json', 'config/scale-profiles/smoke.json',
    'config/reference/airport-anchors.json', 'config/demo_config.json', 'config/feature_flags.json',
    'contracts/schemas/classification-vocabulary.json', 'deployment/artifact-manifest.yaml',
    'deployment/manifest.json', 'data/reference/airports.json', 'data/reference/airlines.json',
    'data/reference/aircraft_types.json', 'warehouse/00_enterprise_schema.sql', 'eventhouse/00_schema.kql',
    'semantic-model/AirportOpsSharedModel.SemanticModel/definition/model.tmdl',
    'reports/AirportOpsPersonaReports.Report/definition.pbir', 'data-agent/definition.json',
    'fabric-app/app-manifest.json', 'digital-twin/dtdl/Airport.json', 'docs/deployment-runbook.md']


def record(check_name, status, detail='', mandatory=True):
    row = {
        'check_name': check_name, 'status': status, 'detail': str(detail)[:4000],
        'mandatory': bool(mandatory), 'environment_name': environment_name,
        'observed_at': datetime.now(timezone.utc), 'is_synthetic': True}
    RESULTS.append(row)
    print(status, check_name, detail)
    return status == 'PASS'


record('environment_name', 'PASS' if environment_name in {'dev', 'test'} else 'FAIL', environment_name)
record('workspace_identifier_format', 'PASS' if dry_run or re.fullmatch(r'[0-9a-fA-F-]{36}', workspace_id) else 'FAIL', 'runtime-only; value not logged')
record('capacity_reference_present', 'PASS' if dry_run or bool(capacity_reference) else 'FAIL', 'runtime-only; value not logged')

missing_paths = [relative_path for relative_path in REQUIRED_PATHS if not (ROOT / relative_path).exists()]
record('required_artifacts', 'PASS' if not missing_paths else 'FAIL', ','.join(missing_paths) or 'complete')

if not missing_paths:
    config = json.loads((ROOT / 'config/base/platform.json').read_text(encoding='utf-8'))
    compatibility_config = json.loads((ROOT / 'config/demo_config.json').read_text(encoding='utf-8'))
    airport_snapshot = json.loads((ROOT / config['reference_snapshot']).read_text(encoding='utf-8'))
    airlines = json.loads((ROOT / 'data/reference/airlines.json').read_text(encoding='utf-8'))['records']
    aircraft_types = json.loads((ROOT / 'data/reference/aircraft_types.json').read_text(encoding='utf-8'))['records']
    classifications = set(json.loads((ROOT / 'contracts/schemas/classification-vocabulary.json').read_text(encoding='utf-8'))['classifications'])
    feature_flags = json.loads((ROOT / 'config/feature_flags.json').read_text(encoding='utf-8'))
    record('classification_vocabulary', 'PASS' if classifications == {'PublicReference','SyntheticMaster','SyntheticOperational','DerivedAnalytical'} else 'FAIL', sorted(classifications))
    record('seed', 'PASS' if config['seed'] == 42 and compatibility_config['random_seed'] == 42 else 'FAIL', config['seed'])
    record('simulation_profile', 'PASS' if config['scale_profile'] in {'unit', 'smoke', 'demo', 'enterprise'} else 'FAIL', config['scale_profile'])
    region_counts = Counter(record['region'] for record in airport_snapshot['records'])
    record('airport_reference_portfolio', 'PASS' if region_counts == Counter({'France':6,'Italy':5,'Portugal':5,'Jordan':2}) else 'FAIL', dict(region_counts))
    record('reference_catalog_counts', 'PASS' if len(airlines) == 20 and len(aircraft_types) == 16 else 'FAIL', {'airlines':len(airlines),'aircraft_types':len(aircraft_types)})
    record('public_reference_classification', 'PASS' if all(not item['is_synthetic'] and item['data_classification']=='PublicReference' for item in airlines + aircraft_types) else 'FAIL', 'airline and aircraft references')
    record('reference_provenance', 'PASS' if airport_snapshot['validation_status'] == 'SOURCE_VERIFIED' and airport_snapshot.get('field_provenance') else 'FAIL', airport_snapshot['validation_status'])
    record('safe_deployment_defaults', 'PASS' if config['deployment_mode'] == 'dry-run' and config['destructive_operations_enabled'] is False and config['external_integrations_enabled'] is False else 'FAIL', 'dry-run/no destructive operations/no external adapters')
    record('compatibility_config', 'PASS' if compatibility_config['airport_count'] == 18 and compatibility_config['simulation_profile'] == 'smoke' else 'FAIL', 'legacy notebook aliases aligned')
    record('rayfin_feature_gate', 'PASS' if feature_flags['features']['enable_rayfin_module'] is False else 'FAIL', 'native path disabled; fallback source only')

for module_name in ['pyspark', 'delta', 'requests']:
    record('python_module_' + module_name, 'PASS' if importlib.util.find_spec(module_name) else 'FAIL', module_name)

if dry_run:
    record('workspace_authorization', 'BLOCKED', 'Dry-run does not acquire a token or call Fabric', mandatory=False)
    record('data_agent_item_capability', 'BLOCKED', 'Requires authenticated target-tenant capability probe', mandatory=False)
    record('fabric_app_publication', 'BLOCKED', 'Requires authenticated target and audience assignment capability', mandatory=False)
    record('rayfin_native_capability', 'UNSUPPORTED', 'No verified native Fabric item type or deployment API', mandatory=False)
else:
    token = notebookutils.credentials.getToken('pbi')
    response = requests.get(
        'https://api.fabric.microsoft.com/v1/workspaces/' + workspace_id,
        headers={'Authorization': 'Bearer ' + token}, timeout=60)
    record('workspace_authorization', 'PASS' if response.status_code == 200 else 'FAIL', 'HTTP ' + str(response.status_code))

try:
    spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('preflight_results')
except Exception as exc:
    print('BLOCKED preflight_results Delta log:', str(exc))

failures = [row for row in RESULTS if row['mandatory'] and row['status'] == 'FAIL']
if failures and strict_mode:
    raise AssertionError('Preflight failed: ' + ', '.join(row['check_name'] for row in failures))
print('Preflight complete:', len(RESULTS), 'checks;', len(failures), 'mandatory failures; dry_run=', dry_run)
